# Collect GVGAI ASCII shards on Colab

Runs the Java MCTS collector (`tracks.singlePlayer.ascii.RunAsciiCollectionMCTS`) on Colab's CPU and writes shards **directly to Google Drive** under the canonical layout shared with `train_ascii_vae_colab.ipynb`:

```
MyDrive/DecoupliWo/data/transitions/{train,test}/<game>/<variant>/shard_XXXXX/
```

## Runtime

- `Runtime` → `Change runtime type` → **CPU** (GPU is wasted here; MCTS is single-threaded CPU-bound)
- More vCPUs = faster collection. Colab High-RAM / Pro runtimes give more cores which `--jobs` scales across.

## One-time prerequisite on Drive

Your local `gvgai/` is an embedded git repo inside DecoupliWo (not a tracked submodule), so `git clone DecoupliWo` on Colab does NOT bring the GVGAI source. You must upload a tarball of your local `gvgai/` directory to Drive **once**. On your Mac:

```bash
cd ~/Documents/GitHub/DecoupliWo
tar -czf gvgai.tar.gz \
  gvgai/src \
  gvgai/examples \
  gvgai/sprites \
  gvgai/out \
  gvgai/gson-2.6.2.jar
```

Verify it contains the compiled collector class:

```bash
tar -tzf gvgai.tar.gz | grep 'RunAsciiCollectionMCTS.class'
# expected: gvgai/out/production/gvgai/tracks/singlePlayer/ascii/RunAsciiCollectionMCTS.class
```

Upload `gvgai.tar.gz` to `MyDrive/DecoupliWo/gvgai.tar.gz`. Cell 4 asserts it exists; Cell 8 extracts it into `/content/DecoupliWo/gvgai/` on each session.

The long-term fix (out of scope here) is to either make `gvgai/` a proper git submodule or push it to its own GitHub repo so it clones automatically.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Canonical paths (shared with the training notebook)

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/wessimpson/DecoupliWo.git'
REPO_BRANCH = 'gameNGen-world_model-v2'
REPO_DIR = Path('/content/DecoupliWo')

DRIVE_ROOT = Path('/content/drive/MyDrive/DecoupliWo')
DRIVE_GVGAI_TAR = DRIVE_ROOT / 'gvgai.tar.gz'
DRIVE_DATA_DIR = DRIVE_ROOT / 'data' / 'transitions'

assert DRIVE_ROOT.parent.is_dir(), (
    f'Expected Drive to be mounted at {DRIVE_ROOT.parent}. '
    f'Re-run the "Mount Drive" cell above.'
)
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_DATA_DIR / 'train').mkdir(parents=True, exist_ok=True)
(DRIVE_DATA_DIR / 'test').mkdir(parents=True, exist_ok=True)

assert DRIVE_GVGAI_TAR.is_file(), (
    f'Expected {DRIVE_GVGAI_TAR} on Drive. See the notebook header for the '
    f'one-time tar command on your laptop.'
)
print('OK gvgai bundle  :', DRIVE_GVGAI_TAR, f'({DRIVE_GVGAI_TAR.stat().st_size / 1e6:.1f} MB)')
print('OK drive data dir:', DRIVE_DATA_DIR)

## 3. Clone the repo

In [ ]:
import subprocess

if REPO_DIR.is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

## 4. Extract the prebuilt GVGAI bundle into the repo

Extracts `MyDrive/DecoupliWo/gvgai.tar.gz` into `/content/DecoupliWo/gvgai/`. The tar includes the already-compiled `out/production/gvgai/**/*.class` so Colab doesn't need `javac`. Idempotent: if the collector class already exists (e.g. you re-ran the cell), it skips.

In [ ]:
import tarfile

GVGAI_DIR = REPO_DIR / 'gvgai'
COLLECTOR_CLASS = GVGAI_DIR / 'out' / 'production' / 'gvgai' / 'tracks' / 'singlePlayer' / 'ascii' / 'RunAsciiCollectionMCTS.class'

if COLLECTOR_CLASS.is_file():
    print(f'already extracted -> {GVGAI_DIR}')
else:
    with tarfile.open(DRIVE_GVGAI_TAR, 'r:gz') as tf:
        tf.extractall(REPO_DIR)
    print(f'extracted -> {GVGAI_DIR}')

assert COLLECTOR_CLASS.is_file(), (
    f'Expected compiled class at {COLLECTOR_CLASS} after extraction. '
    f'Rebuild the tar on your laptop so it includes gvgai/out/production/gvgai/**.'
)
print('OK compiled class:', COLLECTOR_CLASS)

## 5. Verify Java is available

In [ ]:
!java -version
!javac -version || true

If the cell above failed (no JRE on this VM), uncomment the install line in the next cell and re-run. Skip if it already worked.

In [ ]:
# !apt-get update -qq && apt-get install -y -qq default-jre

## 6. Collect

Shards are written **directly to Drive** at `DRIVE_DATA_DIR/<split>/<game>/<variant>/shard_XXXXX/`. The collector is idempotent: each Java run scans the `shard_XXXXX` directories that already exist under its output path and starts numbering from the next free index, so you can kill the cell, restart Colab, and re-run the same config to extend the dataset without clobbering prior shards.

### Tuning knobs (defined at the top of the next cell)

- `GAMES`: comma-separated games with mappings under `world_model/ascii/mappings/<game>.json`.
- `VARIANTS`: VGDL physics variants per game. `stock` wins fast (few wasted ticks); `physics_a|b|c` usually time out at 2000 ticks per episode because MCTS loses.
- `FRAMES_PER_GAME`: total frames per game, split 90/10 train/test, then split evenly across variants.
- `MCTS_MS`: per-tick MCTS budget in ms. Main bottleneck. `40` = stronger play, `20` = ~2× faster, `10` = ~3× faster with noisier trajectories (still fine for VAE training).
- `JOBS`: auto-set to `os.cpu_count()` below. Each `(game, variant, split)` is an independent Java process, so parallelism scales ~linearly until the 24 run slots are filled or vCPUs run out.

### Smoke-test vs production defaults

The next cell ships with a **smoke-test budget** (`stock` variant only, ~40k frames/game, `MCTS_MS=20`) that finishes in ~15–30 min on a 4-vCPU Colab VM and is enough to verify the full pipeline end-to-end. For a production dataset, edit the constants to:

```python
GAMES = 'aliens,chopper,waves'
VARIANTS = 'stock,physics_a,physics_b,physics_c'
FRAMES_PER_GAME = 500_000
MCTS_MS = 40
```

That's ~1.5M frames and takes ~2–8 h depending on vCPU count. Colab sessions time out at ~12 h, so for production prefer to **split collection across sessions by game** (e.g. one session with `GAMES = 'aliens'`, next with `'chopper'`, etc.). Any interrupted run is safe to resume by re-executing this cell — the Java-side `ShardAccumulator` picks up from the next free `shard_XXXXX` index.

In [ ]:
import os

GAMES = 'aliens,chopper,waves'
VARIANTS = 'stock'
FRAMES_PER_GAME = 40_000
MCTS_MS = 20
CHUNK_SIZE = 5000
JOBS = os.cpu_count() or 2

print(f'vCPUs detected: {os.cpu_count()}  ->  --jobs={JOBS}')
print(f'plan          : games=[{GAMES}]  variants=[{VARIANTS}]  {FRAMES_PER_GAME:,} frames/game  mcts_ms={MCTS_MS}')

!python -m data.collect_gvgai_ascii \
  --games {GAMES} \
  --variants {VARIANTS} \
  --frames-per-game {FRAMES_PER_GAME} \
  --mcts-ms {MCTS_MS} \
  --chunk-size {CHUNK_SIZE} \
  --out-root "{DRIVE_DATA_DIR}" \
  --gvgai-root "{GVGAI_DIR}" \
  --jobs {JOBS}

## 7. Summarise what was written

In [ ]:
import numpy as np
from collections import Counter

def summarise(root: Path) -> None:
    per_game = Counter()
    total_frames = 0
    for obs_path in root.rglob('shard_*/obs.npy'):
        game = obs_path.parents[2].name if obs_path.parents[2] != root else obs_path.parent.parent.name
        arr = np.load(obs_path, mmap_mode='r')
        per_game[game] += int(arr.shape[0])
        total_frames += int(arr.shape[0])
    print(f'{root} -> {total_frames:,} frames')
    for g, n in sorted(per_game.items()):
        print(f'  {g:>16}: {n:,}')

summarise(DRIVE_DATA_DIR / 'train')
summarise(DRIVE_DATA_DIR / 'test')